# Whisky Catalogue Pipeline Usage Demo

This notebook demonstrates the library-first API in `src/rs_demo/`.

It uses the existing extracted PDF artifacts when available, then writes demo outputs to `data/demo_notebook/` so the main working files are not overwritten.

In [1]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "src" / "rs_demo").exists():
            return path
    raise RuntimeError("Could not find repo root")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DATA_ROOT = REPO_ROOT / "data"
EXTRACTED_ROOT = DATA_ROOT / "extracted"
PROBE_ROOT = EXTRACTED_ROOT / "pymupdf_probe"
DEMO_ROOT = DATA_ROOT / "demo_notebook"
DEMO_ROOT.mkdir(parents=True, exist_ok=True)

REPO_ROOT

PosixPath('/Users/eltonli/code/rs-demo')

## 1. Import The Library Classes

The reusable pipeline is now exposed as normal Python classes. The package CLI calls these same classes internally.

In [2]:
from rs_demo.catalogue_validation import CatalogueValidationConfig, CatalogueValidator
from rs_demo.embeddings import MockEmbeddingConfig, MockEmbeddingPipeline, NumpyEmbeddingStore
from rs_demo.image_cropping import ImageCropperConfig, ProductImageCropper
from rs_demo.markdown_export import MarkdownExportConfig, ProductMarkdownExporter
from rs_demo.product_parser import ProductParser, ProductParserConfig

"imports ok"

'imports ok'

## 2. Parse Product Records

This step reads `data/extracted/pymupdf_probe/page_XXXX/text.txt` and creates product-level JSONL records. If the extracted page files are missing, run the PDF extraction command first:

```bash
python -m rs_demo extract-pdf data/raw/Great\ Whiskeys.pdf --max-pages 1000
```

In [3]:
demo_catalogue = DEMO_ROOT / "product_parse_sample.jsonl"

records = ProductParser().run(
    ProductParserConfig(
        input_dir=PROBE_ROOT,
        output_path=demo_catalogue,
    )
)

len(records), demo_catalogue

(533,
 PosixPath('/Users/eltonli/code/rs-demo/data/demo_notebook/product_parse_sample.jsonl'))

In [4]:
records[0]

{'product_id': 'p0010-8pm-classic',
 'name': '8PM CLASSIC',
 'brand_or_distillery': '8PM',
 'country': 'India',
 'region': None,
 'style': 'Blend',
 'age': None,
 'abv': None,
 'description': 'Made from “a mix of quality grains,” this has a core that promises “thaath” (boldness, opulence) and “the reach of a man to the dream world.”',
 'brand_description': '8PM had the singular distinction of selling a million cases in its first year (it now sells 3 million). The brand owner is Radico Khaitan, based at Rampur Distillery, Uttar Pradesh. Established in 1943, it is now a gigantic unit with a capacity of over 20 million gallons (90 million liters) of alcohol a year. The company owns other whiskey brands, including Whytehall, and it has recently formed a partnership with Diageo, the world’s largest drinks conglomerate, to produce Masterstroke (see p248).',
 'source_page': 10,
 'page_image_path': '/Users/eltonli/code/rs-demo/data/extracted/pymupdf_probe/page_0010/page_render.png',
 'product_

## 3. Crop Product Images

The cropper updates the catalogue with `cropped_product_image_path` and writes a crop manifest. This can take a little longer than the text-only steps.

In [5]:
crop_manifest = DEMO_ROOT / "product_image_crops_manifest.json"
crop_dir = DEMO_ROOT / "product_image_crops"

cropped_records, crop_results = ProductImageCropper().run(
    ImageCropperConfig(
        catalogue_path=demo_catalogue,
        output_catalogue_path=demo_catalogue,
        output_dir=crop_dir,
        manifest_path=crop_manifest,
        source_root=PROBE_ROOT,
    )
)

sum(1 for result in crop_results if result["status"] == "cropped"), len(crop_results)

(360, 360)

## 4. Validate The Catalogue

The validator compares parsed products with the extraction manifest and writes a Markdown report.

In [6]:
validation_report = DEMO_ROOT / "catalogue_validation_report.md"

report = CatalogueValidator().run(
    CatalogueValidationConfig(
        catalogue_path=demo_catalogue,
        manifest_path=PROBE_ROOT / "manifest.json",
        output_path=validation_report,
    )
)

print("\n".join(report.splitlines()[:18]))
validation_report

# Catalogue Validation Report

## Summary

| Metric | Value |
| --- | --- |
| records | 533 |
| pages in manifest | 386 |
| pages with parsed products | 360 |
| skipped page ranges | 1-9, 62-63, 150-151, 204-205, 332-333, 372-373, 380-386 |
| records missing ABV | 14 |
| records missing country | 77 |
| suspicious names | 1 |

## Parse Confidence

| Confidence | Count |
| --- | --- |


PosixPath('/Users/eltonli/code/rs-demo/data/demo_notebook/catalogue_validation_report.md')

## 5. Generate Manual Review Markdown

This creates one Markdown file per product, plus an index.

In [7]:
markdown_dir = DEMO_ROOT / "product_markdown"

markdown_records = ProductMarkdownExporter().run(
    MarkdownExportConfig(
        catalogue_path=demo_catalogue,
        output_dir=markdown_dir,
    )
)

len(markdown_records), markdown_dir / "index.md"

(533,
 PosixPath('/Users/eltonli/code/rs-demo/data/demo_notebook/product_markdown/index.md'))

## 6. Build Mock Embeddings

The mock embedding pipeline is deterministic and local. It lets us test the embedding storage and retrieval flow before connecting a real multimodal embedding provider.

In [8]:
embedding_dir = DEMO_ROOT / "mock_embeddings"

batch = MockEmbeddingPipeline().run(
    MockEmbeddingConfig(
        input_path=demo_catalogue,
        output_dir=embedding_dir,
        dimension=16,
    )
)

batch.text_embeddings.shape, batch.image_embeddings.shape, batch.multimodal_embeddings.shape

((533, 16), (533, 16), (533, 16))

In [13]:
demo_catalogue

PosixPath('/Users/eltonli/code/rs-demo/data/demo_notebook/product_parse_sample.jsonl')

In [14]:
embedding_dir

PosixPath('/Users/eltonli/code/rs-demo/data/demo_notebook/mock_embeddings')

In [9]:
loaded = NumpyEmbeddingStore(embedding_dir).read()

len(loaded.metadata), loaded.text_embeddings.shape, loaded.image_embeddings.shape, loaded.multimodal_embeddings.shape

(533, (533, 16), (533, 16), (533, 16))

## 7. Tiny Similarity Search Example

This is not a real retrieval evaluation yet. It only proves that generated arrays can be loaded and ranked.

In [10]:
import numpy as np

query_text = "peated smoky Islay whisky with sea salt"
query_text

'peated smoky Islay whisky with sea salt'

In [17]:
loaded.metadata[0]

{'abv': None,
 'age': None,
 'brand_or_distillery': '8PM',
 'country': 'India',
 'image_confidence': 'medium',
 'image_path': '/Users/eltonli/code/rs-demo/data/demo_notebook/product_image_crops/page_0010/embedded_image_01.crop.png',
 'name': '8PM CLASSIC',
 'product_id': 'p0010-8pm-classic',
 'region': None,
 'source_page': 10,
 'style': 'Blend',
 'text': 'Name: 8PM CLASSIC\nBrand: 8PM\nStyle: Blend\nCountry: India\nProduct notes: Made from “a mix of quality grains,” this has a core that promises “thaath” (boldness, opulence) and “the reach of a man to the dream world.”\nBrand context: 8PM had the singular distinction of selling a million cases in its first year (it now sells 3 million). The brand owner is Radico Khaitan, based at Rampur Distillery, Uttar Pradesh. Established in 1943, it is now a gigantic unit with a capacity of over 20 million gallons (90 million liters) of alcohol a year. The company owns other whiskey brands, including Whytehall, and it has recently formed a partner

In [11]:
from rs_demo.embeddings import MockEmbeddingModel

model = MockEmbeddingModel(dimension=loaded.text_embeddings.shape[1])
query = model.embed_text(query_text)
scores = loaded.text_embeddings @ query
top_indices = np.argsort(scores)[-5:][::-1]

[
    {
        "rank": rank,
        "score": float(scores[index]),
        "product_id": loaded.metadata[index]["product_id"],
        "name": loaded.metadata[index].get("name"),
        "style": loaded.metadata[index].get("style"),
    }
    for rank, index in enumerate(top_indices, start=1)
]

[{'rank': 1,
  'score': 0.7851254940032959,
  'product_id': 'p0319-slyrs',
  'name': 'SLYRS',
  'style': 'Single Malt'},
 {'rank': 2,
  'score': 0.7069522142410278,
  'product_id': 'p0208-johnnie-walker-black-label',
  'name': 'JOHNNIE WALKER BLACK LABEL',
  'style': 'Blend'},
 {'rank': 3,
  'score': 0.6697531342506409,
  'product_id': 'p0089-clynelish-14-year-old',
  'name': 'CLYNELISH 14-YEAR-OLD',
  'style': 'Single Malt'},
 {'rank': 4,
  'score': 0.6679806709289551,
  'product_id': 'p0369-the-wild-geese-classic-blend',
  'name': 'THE WILD GEESE CLASSIC BLEND',
  'style': 'Blend'},
 {'rank': 5,
  'score': 0.6490294337272644,
  'product_id': 'p0091-compass-box-the-peat-monster',
  'name': 'COMPASS BOX THE PEAT MONSTER',
  'style': 'Blended Malt'}]

## 8. Equivalent CLI Commands

The notebook uses classes directly. The same workflow is available through the package CLI:

```bash
python -m rs_demo parse-products
python -m rs_demo crop-images
python -m rs_demo validate-catalogue
python -m rs_demo export-markdown
python -m rs_demo build-mock-embeddings
```